In [1]:
import json
import pandas as pd

def check_missing(data, format_type):
    """Check for missing/empty cells in JSON or AVRO data"""
    
    missing = {'total': 0, 'locations': []}
    
    def scan(obj, path=''):
        if obj is None or (isinstance(obj, str) and obj.strip() == ''):
            missing['total'] += 1
            missing['locations'].append(f"{path}: {repr(obj)}")
        elif isinstance(obj, dict):
            for k, v in obj.items():
                scan(v, f"{path}.{k}" if path else k)
        elif isinstance(obj, list):
            for i, item in enumerate(obj):
                scan(item, f"{path}[{i}]")
    
    # Load data
    if format_type.lower() == 'json':
        if isinstance(data, str):
            if data.endswith('.json'):
                with open(data) as f:
                    data = json.load(f)
            else:
                data = json.loads(data)
        scan(data)
    
    elif format_type.lower() == 'avro':
        from avro.datafile import DataFileReader
        from avro.io import DatumReader
        with open(data, 'rb') as f:
            reader = DataFileReader(f, DatumReader())
            for i, record in enumerate(reader):
                scan(record, f'record[{i}]')
            reader.close()
    
    # Report results
    print(f"\n📊 TOTAL MISSING CELLS: {missing['total']}")
    if missing['total'] > 0:
        print("\n📍 LOCATIONS:")
        for loc in missing['locations'][:20]:  # Show first 20
            print(f"  • {loc}")
        if len(missing['locations']) > 20:
            print(f"  ... and {len(missing['locations']) - 20} more")
    else:
        print("✅ No missing cells found")
    return missing['total']

# Examples:
json_data = '{"name": "John", "age": null, "email": "", "city": "NYC"}'
check_missing(json_data, 'json')

# For DataFrame users:
def check_df(df):
    total = df.isnull().sum().sum() + (df.astype(str).eq('').sum().sum())
    print(f"\n📊 TOTAL MISSING CELLS: {total}")
    if total > 0:
        print("\n📍 BY COLUMN:")
        for col in df.columns:
            nulls = df[col].isnull().sum()
            empties = (df[col].astype(str).str.strip() == '').sum()
            if nulls + empties > 0:
                print(f"  • {col}: {nulls} null, {empties} empty")
    return total

# Usage for DataFrame:
# df = pd.DataFrame({'A': [1, None, ''], 'B': ['x', '', None]})
# check_df(df)


📊 TOTAL MISSING CELLS: 2

📍 LOCATIONS:
  • age: None
  • email: ''
